In [19]:
# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
titanic = sns.load_dataset('titanic')

# Function: counts + conditional probabilities
def cond_table(df, by):
    counts = df.groupby([by, 'survived']).size().unstack(fill_value=0)
    counts.columns = ['Died', 'Survived']
    totals = counts.sum(axis=1)
    cond_p = (counts['Survived'] / totals).round(3)
    return pd.DataFrame({
        by: counts.index,
        'Survivors': counts['Survived'],
        'Total': totals,
        'Conditional probability': cond_p
    })

# Tables
gender_tbl = cond_table(titanic, 'sex')
class_tbl = cond_table(titanic, 'pclass').sort_index()
p_table = titanic.pivot_table(values='survived', index='sex', columns='pclass', aggfunc='mean').round(3)
abs_counts = titanic.groupby(['sex', 'pclass', 'survived']).size().unstack(fill_value=0)
abs_counts.columns = ['Died', 'Survived']
# Survival stats by gender and class
survival_stats = titanic.groupby(['sex', 'pclass'])['survived'].agg(Survivors='sum',Total='count').reset_index()
survival_stats['Conditional probability'] = (survival_stats['Survivors'] / survival_stats['Total']).round(3)

# Export all tables to one Excel file
with pd.ExcelWriter('titanic_survival_tables.xlsx') as writer:
    gender_tbl.to_excel(writer, sheet_name='CondProb_by_Gender', index=False)
    class_tbl.to_excel(writer, sheet_name='CondProb_by_Class', index=False)
    p_table.to_excel(writer, sheet_name='Prob_by_Gender_Class')
    abs_counts.to_excel(writer, sheet_name='Absolute_Counts')
    survival_stats.to_excel(writer, sheet_name='Survival_by_Gender_Class', index=False)

# Charts
# Bar plots for gender & class
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.barplot(x='sex', y='survived', data=titanic, estimator=np.mean, errorbar=None, ax=axes[0])
axes[0].set_title('Survival Probability by Gender')
axes[0].set_ylabel('Probability')

sns.barplot(x='pclass', y='survived', data=titanic, estimator=np.mean, errorbar=None, ax=axes[1])
axes[1].set_title('Survival Probability by Class')
axes[1].set_ylabel('Probability')

plt.tight_layout()
plt.savefig('barplots_gender_class.png', dpi=300)
plt.close()

# Grouped bar plot
plt.figure()
sns.barplot(data=titanic, x='pclass', y='survived', hue='sex', estimator=np.mean, errorbar=None)
plt.title('Survival Probability by Class and Gender')
plt.ylabel('Probability')
plt.savefig('grouped_barplot.png', dpi=300)
plt.close()

# Point plot
plt.figure()
sns.pointplot(data=titanic, x='pclass', y='survived', hue='sex',
              estimator=np.mean, dodge=True, markers=['o', 's'], capsize=0.1)
plt.title('Survival Probability Trend by Class and Gender')
plt.ylabel('Probability')
plt.savefig('pointplot.png', dpi=300)
plt.close()

# Heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(p_table, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title('Survival Probability by Gender and Class')
plt.ylabel('Gender')
plt.xlabel('Passenger Class')
plt.savefig('heatmap_gender_class.png', dpi=300)
plt.close()

from google.colab import files
!zip outputs.zip titanic_survival_tables.xlsx *.png
files.download('outputs.zip')

updating: titanic_survival_tables.xlsx (deflated 12%)
updating: barplots_gender_class.png (deflated 30%)
updating: grouped_barplot.png (deflated 26%)
updating: heatmap_gender_class.png (deflated 17%)
updating: pointplot.png (deflated 17%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [44]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import binom

# Simulation parameters
n = 10
p = 0.5
trials = 1000

# Simulate: number of heads per trial
heads = np.random.binomial(n=n, p=p, size=trials)
tails = n - heads

# Chart 1: Heads distribution with theoretical PMF
x = np.arange(0, n + 1)
pmf = binom.pmf(x, n, p)

plt.figure(figsize=(8, 5))
plt.hist(heads, bins=np.arange(n + 2) - 0.5, density=True,
         color='steelblue', edgecolor='black', label='Empirical Distribution')
plt.plot(x, pmf, 'o-', color='darkred', linewidth=2, label='Theoretical PMF')
plt.title('Binomial Distribution: Number of Heads in 10 Coin Tosses (1000 Trials)')
plt.xlabel('Number of Heads per Trial')
plt.ylabel('Probability')
plt.xticks(x)
plt.legend()
plt.tight_layout()
plt.savefig('binomial_heads_distribution.png', dpi=300, bbox_inches='tight')
plt.close()

# Chart 2: Total percentage of heads vs tails across all trials
total_heads = np.sum(heads)
total_tails = np.sum(tails)
percent_heads = total_heads / (trials * n) * 100
percent_tails = total_tails / (trials * n) * 100

plt.figure(figsize=(6, 5))
plt.bar(['Heads', 'Tails'], [percent_heads, percent_tails],
        color=['steelblue', 'lightcoral'], edgecolor='black')
plt.title('Total Percentage of Heads vs Tails (1000 Trials × 10 Tosses)')
plt.ylabel('Percentage (%)')
plt.ylim(0, 100)
plt.tight_layout()
plt.savefig('heads_tails_percentage.png', dpi=300, bbox_inches='tight')
plt.close()

# Table: Heads and Tails per trial
df = pd.DataFrame({'Trial': np.arange(1, trials + 1), 'Heads': heads, 'Tails': tails})
df.to_excel('heads_tails_table.xlsx', index=False)

# Download files
from google.colab import files
files.download('binomial_heads_distribution.png')
files.download('heads_tails_percentage.png')
files.download('heads_tails_table.xlsx')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from google.colab import files

# Datasets to scan
dataset_names = [
    "iris", "penguins", "mpg", "tips", "diamonds", "planets",
    "flights", "exercise", "fmri", "dots"
]

# Load datasets safely
series_catalog = []  # list of dicts: {key, dataset, column, series, mean, var, std, cv}
for name in dataset_names:
    try:
        df = sns.load_dataset(name)
    except Exception:
        continue
    # Keep only numeric columns
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            s = df[col].dropna()
            if len(s) < 30:  # ensure stable estimates
                continue
            mean = s.mean()
            std = s.std()
            var = s.var()
            if mean == 0 or np.isnan(mean) or np.isnan(std) or var == 0:
                continue
            series_catalog.append({
                "key": f"{name}.{col}",
                "dataset": name,
                "column": col,
                "series": s,
                "mean": mean,
                "std": std,
                "var": var,
                "cv": std / mean
            })

# Find pairs with similar variance but different CV
candidates = []
for a, b in combinations(series_catalog, 2):
    # Relative variance difference
    vA, vB = a["var"], b["var"]
    rel_var_diff = abs(vA - vB) / max(vA, vB)
    if rel_var_diff > 0.10:  # tighten/relax this tolerance as needed (10%)
        continue
    # Demand different means and CVs
    mean_ratio = max(a["mean"], b["mean"]) / min(a["mean"], b["mean"])
    cv_diff = abs(a["cv"] - b["cv"])
    if mean_ratio < 1.5:  # means should be meaningfully different
        continue
    if cv_diff < 0.15:    # CVs should differ meaningfully
        continue
    candidates.append({
        "A": a, "B": b,
        "rel_var_diff": rel_var_diff,
        "mean_ratio": mean_ratio,
        "cv_diff": cv_diff
    })

# If nothing found, relax thresholds and retry
if not candidates:
    # Second pass with looser thresholds
    for a, b in combinations(series_catalog, 2):
        vA, vB = a["var"], b["var"]
        rel_var_diff = abs(vA - vB) / max(vA, vB)
        if rel_var_diff > 0.20:
            continue
        mean_ratio = max(a["mean"], b["mean"]) / min(a["mean"], b["mean"])
        cv_diff = abs(a["cv"] - b["cv"])
        if mean_ratio < 1.3:
            continue
        if cv_diff < 0.10:
            continue
        candidates.append({
            "A": a, "B": b,
            "rel_var_diff": rel_var_diff,
            "mean_ratio": mean_ratio,
            "cv_diff": cv_diff
        })

# Pick the best candidate: closest variance, largest CV difference
if not candidates:
    raise ValueError("No suitable pair found. Try adding datasets or loosening thresholds.")

candidates.sort(key=lambda x: (x["rel_var_diff"], -x["cv_diff"], -x["mean_ratio"]))
best = candidates[0]
A, B = best["A"], best["B"]

# Build results table
df_out = pd.DataFrame([
    {"Dataset.Column": A["key"], "Mean": A["mean"], "Variance": A["var"], "CV": A["cv"]},
    {"Dataset.Column": B["key"], "Mean": B["mean"], "Variance": B["var"], "CV": B["cv"]},
])

# Save table
df_out.to_csv("similar_variance_diff_cv.csv", index=False)

# Plot: show both variance and CV to make the point (but don't display)
fig, axes = plt.subplots(1, 2, figsize=(11,4))

# Variance bars
axes[0].bar([A["key"], B["key"]], [A["var"], B["var"]], color=["steelblue", "orange"])
axes[0].set_title("Variance (similar)")
axes[0].set_ylabel("Variance")
for i, v in enumerate([A["var"], B["var"]]):
    axes[0].text(i, v * 1.02, f"{v:.2f}", ha="center")

# CV bars
axes[1].bar([A["key"], B["key"]], [A["cv"], B["cv"]], color=["steelblue", "orange"])
axes[1].set_title("Coefficient of Variation (different)")
axes[1].set_ylabel("CV")
for i, v in enumerate([A["cv"], B["cv"]]):
    axes[1].text(i, v * 1.02, f"{v:.2f}", ha="center")

plt.tight_layout()
plt.savefig("similar_variance_diff_cv.png", dpi=300)
plt.close(fig)  # do not display

# Download both files (Colab)
files.download("similar_variance_diff_cv.csv")
files.download("similar_variance_diff_cv.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>